# Recipe RAG System

A RAG (Retrieval-Augmented Generation) system for recipe discovery and cooking assistance.

## Architecture

1. **Data Pipeline** - Load, parse, and validate recipe data from Kaggle
2. **RAG Core** - Vector embeddings with ChromaDB for semantic search
3. **App** - LLM-powered chat interface with Gradio

## Setup

Create a `.env` file with your OpenAI API key:
```
OPENAI_API_KEY=your-key-here
```

In [ ]:
# Install dependencies
!pip install kagglehub chromadb sentence-transformers gradio openai python-dotenv pandas -q

## Step 1: Load and Process Data

The `DataPipeline` handles loading from Kaggle, parsing ingredients/instructions, and validation.

In [ ]:
from data_pipeline import DataPipeline
import json

# Run the full pipeline: load -> parse -> validate -> save
pipeline = DataPipeline()
validated_recipes = pipeline.run(num_recipes=100)

# Show a sample recipe
print("\nSample recipe:")
sample = validated_recipes[0]
print(f"Title: {sample['title']}")
print(f"Cuisine: {sample.get('cuisine', 'Unknown')}")
print(f"Prep: {sample.get('prep_time')} min | Cook: {sample.get('cook_time')} min")
print(f"Ingredients: {len(sample['ingredients'])}")
print(f"Dietary: {json.dumps(sample.get('dietary_info', {}), indent=2)}")

## Step 2: Create Vector Store

The `RecipeRAG` class creates embeddings and stores them in ChromaDB for semantic search.

In [ ]:
from rag_core import RecipeRAG

# Initialize RAG and add recipes
rag = RecipeRAG(validated_recipes)
rag.add_recipes(validated_recipes)

print(f"\nStats: {rag.get_stats()}")

## Step 3: Test Search

Test different search capabilities before launching the chat interface.

In [ ]:
# Test ingredient-based search
print("Ingredient search (chicken):")
results = rag.search_by_ingredients(['chicken'], max_results=3)
for r in results:
    print(f"  - {r['recipe']['title']} (match: {r['ingredient_match']:.2f})")

# Test natural language search
print("\nNatural language search (healthy vegetarian):")
results = rag.search_natural_language("healthy vegetarian dinner", max_results=3)
for r in results:
    print(f"  - {r['recipe']['title']} (score: {r['score']:.2f})")

# Test time-based search
print("\nTime-based search (under 30 min prep):")
results = rag.search_by_time(max_prep_time=30, max_results=3)
for r in results:
    print(f"  - {r['recipe']['title']} (prep: {r['recipe'].get('prep_time')} min)")

## Step 4: Launch Chat Interface

The `RecipeApp` combines LLM prompts with the Gradio UI for an interactive cooking assistant.

In [ ]:
from app import RecipeApp
import gradio as gr

# Close any existing Gradio instances
gr.close_all()

# Create and launch the app
recipe_app = RecipeApp(rag)
recipe_app.launch()